# Chain Reaction — Jane Street, July 2014


[Puzzle page](https://www.janestreet.com/puzzles/chain-reaction-index/)

## Answer

Length: **77**

Chain:

```
 26   52   13   39   78    6   48   24   96   32   64
 16   80   40   20  100   50   25   75   15   45   90
 30   10   60   12   72   36   18   54   27   81    9
 63   21   42   84   28   56    8   88   44   22   66
 33   99   11   77    7   49   98   14   70   35    5
 85   17   34   68    4   76   38   19   57    3   69
 23   92   46    2   58   29   87    1   93   31   62
```


## AI disclaimer

I asked Claude to code this one up as a **walkthrough** rather than a bare solution: almost all of the
code is Claude's. The writing is mine and the goal was comprehension of the solution so I can solve similar problems myself in the future.

## 1. What kind of problem is this?

This is a graph longest path problem.  We construct a graph by created 100 vertices and connecting each vertice to any number it divides or can be divided by.  Then, our problem is to find the longest traversal that does not repeat a vertice. 

Thinking of it as a graph allows us to reason about the problem quite well to help prune the solution space.  


The plan:

1. Reason about the chain by considering the graphs structure to help establish an upper bound and tight constraints
2. Build a greedy chain — a lower bound
3. hand the exact problem to CP-SAT

In [2]:
from ortools.sat.python import cp_model

### Helper functions

Two numbers may sit next to each other exactly when one divides the other. That rule is the only
piece of the puzzle, so it gets written once, in one function, and everything else calls it.

In [11]:
LARGEST = 100
NUMBERS = list(range(1, LARGEST + 1))

EXAMPLE_CHAIN = [37, 74, 2, 8, 4, 16, 48, 6, 3, 9, 27, 81]


def divides_either_way(x, y):
    if y % x == 0:
        return True
    if x % y == 0:
        return True
    return False


def neighbours_of(x):
    neighbours = []
    for y in NUMBERS:
        if y != x and divides_either_way(x, y):
            neighbours.append(y)
    return neighbours


NEIGHBOURS = {
    x: neighbours_of(x) for x in NUMBERS
}  # dictionary for looking up a number's neighbours

EDGES = []
for x in NUMBERS:
    for y in NEIGHBOURS[x]:
        if x < y:
            EDGES.append((x, y))

print(len(NUMBERS), "numbers and", len(EDGES), "legal links between them")

100 numbers and 382 legal links between them


With onlly 382 edges, this problem could be a lot worse!

### Structure

We can make some observations about the graph.

1. Primes over 50 `(53, 59, 61, 67, 71, 73, 79, 83, 89, and 97)` can only sit next to 1.  This means that at most 1 odd prime over 50 can be on our chain, at the beginning or end.  We lose 9 numbers
2. Primes over 33 are restricted to only 2 neighbors (1 and double itself)
3. Numbers over 50 must be non-adjacent.  I.E. we can never have two of them in a row.
4. Odd Numbers over 50 must be divisible by odd numbers under 33.  This means the set `[1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33]` has to handle that work load.
3. Odd numbers over 50 are also very restricted -- can only sit between two odd numbers that are factors of itself and 1.  


In [3]:
def show_least_connected(count):
    """Print the `count` numbers with the fewest legal neighbours, and who those neighbours are."""
    by_degree = sorted(NUMBERS, key=lambda x: len(NEIGHBOURS[x]))
    for x in by_degree[:count]:
        print(f"{x:>3}  degree {len(NEIGHBOURS[x]):>2}   {NEIGHBOURS[x]}")


show_least_connected(20)

 53  degree  1   [1]
 59  degree  1   [1]
 61  degree  1   [1]
 67  degree  1   [1]
 71  degree  1   [1]
 73  degree  1   [1]
 79  degree  1   [1]
 83  degree  1   [1]
 89  degree  1   [1]
 97  degree  1   [1]
 37  degree  2   [1, 74]
 41  degree  2   [1, 82]
 43  degree  2   [1, 86]
 47  degree  2   [1, 94]
 29  degree  3   [1, 58, 87]
 31  degree  3   [1, 62, 93]
 49  degree  3   [1, 7, 98]
 51  degree  3   [1, 3, 17]
 55  degree  3   [1, 5, 11]
 57  degree  3   [1, 3, 19]


## 3. Checker

We will test it on the example chain.

In [8]:
def check(chain):
    for value in chain:
        if value < 1 or value > LARGEST:
            return False

    already_seen = set()
    for value in chain:
        if value in already_seen:
            return False
        already_seen.add(value)

    for position in range(len(chain) - 1):
        x = chain[position]
        y = chain[position + 1]
        if not divides_either_way(x, y):
            return False

    return True

In [9]:
print("the puzzle's example :", str(check(EXAMPLE_CHAIN)))
print("a repeated number    :", str(check([2, 4, 2])))
print("a broken link        :", str(check([2, 4, 6])))
print("out of range         :", str(check([2, 200])))

the puzzle's example : True
a repeated number    : False
a broken link        : False
out of range         : False


## 6. The model: longest path via `add_circuit`

CP-SAT has `add_circuit` a dedicated propagator that reasons about connectivity directly.  As input, you hand it a list of arcs, each a triple `(tail, head, literal)`. It enforces that the arcs whose literals are true form exactly one circuit covering the nodes.


So, we will effectively just pass it all the edges of our graph as x,y and y,x pairs (to indicate either direction works), with a variable to indicate if the edge is used or not.  Then, our model will set that variable to maximize the total length of the circuit.  In order to do that, however, we need to encode the constaints of the problem as well as give the model a function to maximize.

### The modeling tricks

#### Self edges
In order to give the model a function to optimize, we will also create a variable `is_used(x)` that creates an arc `(x, x, literal)` from each number to itself, where the literal is set to `is_used(x).negated()`.  `add_circuit` enforces that all nodes exist on on the cycle exactly once OR are a self loop, so setting is_used(x) to False is the same as removing it from the circuit.     

We obtain an objective function to maximize by summing is_used(x) for x in numbers.

#### Dummy edge
Since `add_circuit` creates a closed loop as a solution, and we want a chain, we will add a dummy edge connecting node 0 to every other number, so it's effectively free.  Then when the solution comes back, we simply chop away the dummy node to get the chain.  

#### Avoiding multiple subpaths

We need to avoid a soution of a subpath of 10, 40, 15, and 8 being read as a solution of 73 since 73 numbers are used.  `add_circuit` enforces this directly, so no work is needed on our end.  Every node can only be part of the circuit or a self loop.

## Specifications

### Variables

All variables are booleans. These are what the model will find --> it will optimize its assignments to maximize our objective function.

`edge_x_y` for x in numbers, y in x_neighbors
`edge_y_x` for x in numbers, y in x_neighbors
`dummy_0_x` for x in numbers
`dummy_x_0` for x in numbers
`is_used(x)` for x in numbers


### Arcs

Arcs are constructed from the variables themselves and are in the shape of `("in", "out", literal)` where literal is True or False depending on if the arc exists in the solution.  These arcs are what we actually pass into `add_circuit`.

`(x, y, edge_x_y)`
`(y, x, edge_y_x)`
`(0, x, dummy_0_x)`
`(x, 0, dummy_x_0)`
`(x, x, is_used(x).negated())`

Since we have 100 numbers with 382 links between them, we have 382 + 382 + 100 + 100 + 100 = 1064 variables.

### Constraints

1. x must appear exactly once as "in" and exactly once as "out" for all x in numbers. (handled by `add_circuit` already).

### Objective function

We maximize the sum of is_used(x).

In [ ]:
DUMMY = 0


def build_model():
    model = cp_model.CpModel()

    # used[x] is true when x appears somewhere in the chain.
    used = {}
    for x in NUMBERS:
        used[x] = model.new_bool_var(f"used_{x}")

    arcs = []  # (tail, head, literal) triples handed to add_circuit
    arc_literal = (
        {}
    )  # (tail, head) -> literal, so the chain can be read back afterwards

    for x in NUMBERS:
        # A true self-loop is how add_circuit is told to leave this node out altogether.
        arcs.append((x, x, used[x].negated()))

        # The dummy node stands for "off the end of the chain": dummy -> x means the chain
        # starts at x, and x -> dummy means it finishes there.
        starts_here = model.new_bool_var(f"starts_{x}")
        ends_here = model.new_bool_var(f"ends_{x}")
        arcs.append((DUMMY, x, starts_here))
        arcs.append((x, DUMMY, ends_here))
        arc_literal[(DUMMY, x)] = starts_here
        arc_literal[(x, DUMMY)] = ends_here

    # Every legal link becomes two arcs, because the chain may traverse it in either direction.
    for x, y in EDGES:
        forwards = model.new_bool_var(f"arc_{x}_{y}")
        backwards = model.new_bool_var(f"arc_{y}_{x}")
        arcs.append((x, y, forwards))
        arcs.append((y, x, backwards))
        arc_literal[(x, y)] = forwards
        arc_literal[(y, x)] = backwards

    model.add_circuit(arcs)
    return model, used, arc_literal

In [13]:
def chain_from_solution(solver, arc_literal):
    """Read a solved model's arcs back as the chain, in order, with the dummy removed."""
    follows = {}
    for (tail, head), literal in arc_literal.items():
        if solver.value(literal) == 1:
            follows[tail] = head

    chain = []
    current = follows[DUMMY]  # the arc leaving the dummy points at the first number
    while current != DUMMY:
        chain.append(current)
        current = follows[current]
    return chain


def find_longest_chain(time_limit_seconds=60):
    """The longest legal chain, as (length, chain). Length is -1 when the requirements are impossible.

    `add_requirements` is an optional function(model, used) used in section 8 to ask
    "what is the longest chain that also does X?".
    """
    model, used, arc_literal = build_model()

    numbers_used = []
    for x in NUMBERS:
        numbers_used.append(used[x])
    model.maximize(sum(numbers_used))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit_seconds
    solver.parameters.num_workers = 8
    status = solver.solve(model)

    print("status         :", solver.status_name(status))
    print("best chain     :", solver.objective_value)
    print("proven ceiling :", solver.best_objective_bound)
    print("wall time      :", round(solver.wall_time, 2), "seconds")

    if status != cp_model.OPTIMAL and status != cp_model.FEASIBLE:
        return -1, []

    chain = chain_from_solution(solver, arc_literal)
    return len(chain), chain

## 7. Solution

In [ ]:
best_length, best_chain = find_longest_chain()

print()
print("longest chain:", best_length, "numbers")
print(best_chain)
print("legal?", check(best_chain))

status         : OPTIMAL
best chain     : 77.0
proven ceiling : 77.0
wall time      : 0.11 seconds

longest chain: 77 numbers
[26, 52, 13, 39, 78, 6, 48, 24, 96, 32, 64, 16, 80, 40, 20, 100, 50, 25, 75, 15, 45, 90, 30, 10, 60, 12, 72, 36, 18, 54, 27, 81, 9, 63, 21, 42, 84, 28, 56, 8, 88, 44, 22, 66, 33, 99, 11, 77, 7, 49, 98, 14, 70, 35, 5, 85, 17, 34, 68, 4, 76, 38, 19, 57, 3, 69, 23, 92, 46, 2, 58, 29, 87, 1, 93, 31, 62]
legal? True


We get a chain of length **77** proven optimal!  It only took about .1 second, which really surprised me but the problem is very heavily constrained because the graph is so sparse of edges.

One thing worth noting is that there are many different solutions, and this is just one of them.  To see that, take any number in the chain that divides or is divisible by the starting number.  Remove the chain from that point on, flip it, and append to the left of the starting number and voila, you have a new length 77 chain. 

### What I take away from this one

- **Recognising the shape of a problem is most of the work.** We needed to recognize that this was a graph problem with a longest path.  This quickly allowed us to find the best tool for the job and the correct encoding of the problem.  
- **`add_circuit` with self-loops and a dummy node** is the pattern for any "longest / best path or
  tour over an optional subset" question. Worth remembering.  I'm hoping as I keep working through these puzzles, I get to start reusing code more often.